In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer

In [2]:
data = pd.read_csv('Bengaluru_House_Data.csv')

In [3]:
data.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [4]:
data.shape

(13320, 9)

In [5]:
data.drop(columns=['area_type','availability','society'], inplace=True)
data.dropna(inplace=True)

In [6]:
data

,location,size,total_sqft,bath,balcony,price
0,Electronic City Phase II,2 BHK,1056,2.0,1.0,39.07
1,Chikka Tirupathi,4 Bedroom,2600,5.0,3.0,120.00
2,Uttarahalli,3 BHK,1440,2.0,3.0,62.00
3,Lingadheeranahalli,3 BHK,1521,3.0,1.0,95.00
4,Kothanur,2 BHK,1200,2.0,1.0,51.00
...,...,...,...,...,...,...
13314,Green Glen Layout,3 BHK,1715,3.0,3.0,112.00
13315,Whitefield,5 Bedroom,3453,4.0,0.0,231.00
13317,Raja Rajeshwari Nagar,2 BHK,1141,2.0,1.0,60.00
13318,Padmanabhanagar,4 BHK,4689,4.0,1.0,488.00


In [7]:
data['location']= data['location'].apply(lambda x:x.strip())
print(data.location.value_counts())

location
Whitefield                         515
Sarjapur  Road                     372
Electronic City                    302
Kanakpura Road                     261
Thanisandra                        234
                                  ... 
MM Layout                            1
beml layout, basaveshwara nagar      1
Kanakapur main road                  1
Prasanna layout Herohalli            1
Chikkajala                           1
Name: count, Length: 1254, dtype: int64


In [8]:
location_stats = data.groupby('location')['location'].agg('count').sort_values(ascending=False)
location_less_than_10_entries = location_stats[location_stats <=10]
location_less_than_10_entries

location
Basapura                   10
Gunjur Palya               10
Dairy Circle               10
HAL 2nd Stage              10
Kalkere                    10
                           ..
mvj engineering college     1
manyata tech park           1
manyata                     1
kg halli jalhalli west      1
kanakapura road             1
Name: location, Length: 1017, dtype: int64

In [9]:
data['location'] = data['location'].apply(lambda x:'other' if x in location_less_than_10_entries else x)
data['location'].value_counts()

location
other               2739
Whitefield           515
Sarjapur  Road       372
Electronic City      302
Kanakpura Road       261
                    ... 
Marsur                11
Karuna Nagar          11
Prithvi Layout        11
Doddaballapur         11
Thyagaraja Nagar      11
Name: count, Length: 238, dtype: int64

In [10]:
data['size'].unique()

array(['2 BHK', '4 Bedroom', '3 BHK', '3 Bedroom', '1 BHK', '1 RK',
       '4 BHK', '1 Bedroom', '2 Bedroom', '6 Bedroom', '8 Bedroom',
       '7 Bedroom', '5 BHK', '7 BHK', '6 BHK', '5 Bedroom', '11 BHK',
       '9 BHK', '9 Bedroom', '27 BHK', '11 Bedroom', '43 Bedroom',
       '14 BHK', '8 BHK', '12 Bedroom', '10 Bedroom', '13 BHK'],
      dtype=object)

In [11]:
data['bedrooms'] = data['size'].apply(lambda x:int(x.split(' ')[0]))
data

,location,size,total_sqft,bath,balcony,price,bedrooms
0,Electronic City Phase II,2 BHK,1056,2.0,1.0,39.07,2
1,Chikka Tirupathi,4 Bedroom,2600,5.0,3.0,120.00,4
2,Uttarahalli,3 BHK,1440,2.0,3.0,62.00,3
3,Lingadheeranahalli,3 BHK,1521,3.0,1.0,95.00,3
4,Kothanur,2 BHK,1200,2.0,1.0,51.00,2
...,...,...,...,...,...,...,...
13314,Green Glen Layout,3 BHK,1715,3.0,3.0,112.00,3
13315,Whitefield,5 Bedroom,3453,4.0,0.0,231.00,5
13317,Raja Rajeshwari Nagar,2 BHK,1141,2.0,1.0,60.00,2
13318,Padmanabhanagar,4 BHK,4689,4.0,1.0,488.00,4


In [12]:
data.total_sqft.unique()

array(['1056', '2600', '1440', ..., '1133 - 1384', '774', '4689'],
      shape=(1976,), dtype=object)

In [13]:
def clean(sqft):
    tokens = sqft.split('-')
    if len (tokens)== 2:
        return(float(tokens[0])+ float(tokens[1]))/2
    else:
        try:
            return float(sqft)
        except:
            return None

In [14]:
data['total_sqft'] = data['total_sqft'].apply(clean)
data['total_sqft'].unique()

array([1056. , 2600. , 1440. , ..., 1258.5,  774. , 4689. ], shape=(1887,))

In [15]:
data.dropna(inplace=True)
data.describe()

,total_sqft,bath,balcony,price,bedrooms
count,12668.000000,12668.000000,12668.000000,12668.000000,12668.000000
mean,1511.835167,2.616277,1.585649,105.952648,2.736422
std,1162.097276,1.223882,0.816758,131.813137,1.202643
min,5.000000,1.000000,0.000000,8.000000,1.000000
25%,1100.000000,2.000000,1.000000,49.015000,2.000000
50%,1260.000000,2.000000,2.000000,70.000000,3.000000
75%,1640.000000,3.000000,2.000000,115.000000,3.000000
max,52272.000000,40.000000,3.000000,2912.000000,43.000000


In [16]:
data['sqft_per_bed'] = data['total_sqft']/data['bedrooms']
data.sqft_per_bed.describe()

count    12668.000000
mean       570.060291
std        380.298999
min          0.714286
25%        473.333333
50%        550.000000
75%        622.500000
max      26136.000000
Name: sqft_per_bed, dtype: float64

In [17]:
data2 = data[data['sqft_per_bed'] >= 300] 
data2

,location,size,total_sqft,bath,balcony,price,bedrooms,sqft_per_bed
0,Electronic City Phase II,2 BHK,1056.0,2.0,1.0,39.07,2,528.000000
1,Chikka Tirupathi,4 Bedroom,2600.0,5.0,3.0,120.00,4,650.000000
2,Uttarahalli,3 BHK,1440.0,2.0,3.0,62.00,3,480.000000
3,Lingadheeranahalli,3 BHK,1521.0,3.0,1.0,95.00,3,507.000000
4,Kothanur,2 BHK,1200.0,2.0,1.0,51.00,2,600.000000
...,...,...,...,...,...,...,...,...
13314,Green Glen Layout,3 BHK,1715.0,3.0,3.0,112.00,3,571.666667
13315,Whitefield,5 Bedroom,3453.0,4.0,0.0,231.00,5,690.600000
13317,Raja Rajeshwari Nagar,2 BHK,1141.0,2.0,1.0,60.00,2,570.500000
13318,Padmanabhanagar,4 BHK,4689.0,4.0,1.0,488.00,4,1172.250000


In [18]:
data2['price_per_sqft']=round(data2['price']*100000/data2['total_sqft'],2)
print(data2.price_per_sqft.describe())

count     12013.000000
mean       6206.082361
std        3985.518849
min         267.830000
25%        4199.360000
50%        5252.530000
75%        6823.530000
max      176470.590000
Name: price_per_sqft, dtype: float64


C:\Users\dthon\AppData\Local\Temp\ipykernel_9616\771491425.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2['price_per_sqft']=round(data2['price']*100000/data2['total_sqft'],2)


In [19]:
data3 = data2[data2['price_per_sqft'] >=2000]
data3.drop(columns = ['size','sqft_per_bed','price_per_sqft'],axis =1, inplace=True)
data3

C:\Users\dthon\AppData\Local\Temp\ipykernel_9616\3468292078.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data3.drop(columns = ['size','sqft_per_bed','price_per_sqft'],axis =1, inplace=True)


,location,total_sqft,bath,balcony,price,bedrooms
0,Electronic City Phase II,1056.0,2.0,1.0,39.07,2
1,Chikka Tirupathi,2600.0,5.0,3.0,120.00,4
2,Uttarahalli,1440.0,2.0,3.0,62.00,3
3,Lingadheeranahalli,1521.0,3.0,1.0,95.00,3
4,Kothanur,1200.0,2.0,1.0,51.00,2
...,...,...,...,...,...,...
13314,Green Glen Layout,1715.0,3.0,3.0,112.00,3
13315,Whitefield,3453.0,4.0,0.0,231.00,5
13317,Raja Rajeshwari Nagar,1141.0,2.0,1.0,60.00,2
13318,Padmanabhanagar,4689.0,4.0,1.0,488.00,4


In [20]:
col_trans=make_column_transformer((OneHotEncoder(sparse_output=False),['location']), remainder='passthrough')
scaler=StandardScaler()
lr=LinearRegression()
model=make_pipeline(col_trans,scaler,lr)
#

In [21]:
dtf_input=data3.drop(columns=['price'])
dtf_output=data3['price']


In [22]:
x_train,x_test,y_train,y_test=train_test_split(dtf_input,dtf_output,test_size=0.2)
model.fit(x_train,y_train)
print(model.score(x_test,y_test))


0.6003039928814891


In [23]:
model.score(x_test,y_test)


0.6003039928814891

In [24]:
input=pd.DataFrame([['Electronic City Phase II','2000','2.0','1.0','3']],columns=['location','total_sqft','bath','balcony','bedrooms'])
print(model.predict(input))

[139.72455329]


In [25]:
import joblib
joblib.dump(model, "house_price_prediction.joblib")
data3.to_csv('cleaned_house_price_prediction.csv', index=False)
